In [1]:
# --- [CELL 0]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 1}
import datetime as dt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

import warnings
warnings.simplefilter(action="ignore")

pd.set_option('display.max_columns',1000)
pd.set_option('display.width', 500)
pd.set_option('display.float_format',lambda x : '%.2f' % x)

In [2]:
# --- [CELL 1]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 2}
df_ = pd.read_csv("data/dataset.csv", compression="gzip")
df = df_.copy()
df.head()

,RecipeId,Name,CookTime,PrepTime,TotalTime,RecipeIngredientParts,Calories,FatContent,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeInstructions
0,38,Low-Fat Berry Blue Frozen Dessert,1440,45,1485,"c(""blueberries"", ""granulated sugar"", ""vanilla ...",170.90,2.50,1.30,8.00,29.80,37.10,3.60,30.20,3.20,"c(""Toss 2 cups berries with sugar."", ""Let stan..."
1,41,Carina's Tofu-Vegetable Kebabs,20,1440,1460,"c(""extra firm tofu"", ""eggplant"", ""zucchini"", ""...",536.10,24.00,3.80,0.00,1558.60,64.20,17.30,32.10,29.30,"c(""Drain the tofu, carefully squeezing out exc..."
2,42,Cabbage Soup,30,20,50,"c(""plain tomato juice"", ""cabbage"", ""onion"", ""c...",103.60,0.40,0.10,0.00,959.30,25.10,4.80,17.70,4.30,"c(""Mix everything together and bring to a boil..."
3,45,Buttermilk Pie With Gingersnap Crumb Crust,50,30,80,"c(""sugar"", ""margarine"", ""egg"", ""flour"", ""salt""...",228.00,7.10,1.70,24.50,281.80,37.50,0.50,24.70,4.20,"c(""Preheat oven to 350°F."", ""Make pie crust, u..."
4,46,A Jad - Cucumber Pickle,0,25,25,"c(""rice vinegar"", ""haeo"")",4.30,0.00,0.00,0.00,0.70,1.10,0.20,0.20,0.10,"c(""Slice the cucumber in four lengthwise, then..."


In [3]:
# --- [CELL 2]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 3}
def grab_col_names(dataframe, cat_th=10, car_th=20):

    cat_cols = [col for col in dataframe.columns if dataframe[col].dtypes == "O"]
    num_but_cat = [col for col in dataframe.columns if dataframe[col].nunique() < cat_th and
                   dataframe[col].dtypes != "O"]
    cat_but_car = [col for col in dataframe.columns if dataframe[col].nunique() > car_th and
                   dataframe[col].dtypes == "O"]
    cat_cols = cat_cols + num_but_cat
    cat_cols = [col for col in cat_cols if col not in cat_but_car]

    # num_cols
    num_cols = [col for col in dataframe.columns if dataframe[col].dtypes != "O"]
    num_cols = [col for col in num_cols if col not in num_but_cat]

    print(f"Observations: {dataframe.shape[0]}")
    print(f"Variables: {dataframe.shape[1]}")
    print(f'cat_cols: {len(cat_cols)}')
    print(f'num_cols: {len(num_cols)}')
    print(f'cat_but_car: {len(cat_but_car)}')
    print(f'num_but_cat: {len(num_but_cat)}')
    return cat_cols, num_cols, cat_but_car

In [4]:
# --- [CELL 3]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 4}
cat_cols, num_cols, num_but_cat = grab_col_names(df)

Observations: 375703
Variables: 16
cat_cols: 0
num_cols: 13
cat_but_car: 3
num_but_cat: 0


In [5]:
# --- [CELL 4]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 5}
def outlier_thresholds(dataframe, col_name, q1=0.01, q3=0.99):
    quartile1= dataframe[col_name].quantile(q1)
    quartile3= dataframe[col_name].quantile(q3)
    interquantile_range = quartile3 -quartile1
    up_limit= quartile3 +1.5 * interquantile_range
    low_limit= quartile1 -1.5 * interquantile_range
    return low_limit, up_limit

In [6]:
# --- [CELL 5]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 6}
def replace_with_thresholds(dataframe, variable):
    low_limit, up_limit = outlier_thresholds(dataframe, variable)
    dataframe.loc[(dataframe[variable] < low_limit), variable] = low_limit
    dataframe.loc[(dataframe[variable] > up_limit), variable] = up_limit

for col in num_cols:
    replace_with_thresholds(df, col)

In [7]:
# --- [CELL 6]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 7}
def check_outlier(dataframe, col_name):
    low_limit, up_limit = outlier_thresholds(dataframe, col_name)
    if dataframe[(dataframe[col_name] > up_limit) | (dataframe[col_name] < low_limit)].any(axis=None):
        return True
    else:
        return False

check_outlier(df,num_cols)

False

In [8]:
# --- [CELL 7]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 8}
df= df.iloc[:,1:]

In [9]:
# --- [CELL 8]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 9}
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from yellowbrick.cluster import KElbowVisualizer
from scipy.cluster.hierarchy import linkage
from scipy.cluster.hierarchy import dendrogram
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import AgglomerativeClustering

In [10]:
# --- [CELL 9]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 10}
cat_cols, num_cols, num_but_cat = grab_col_names(df)

Observations: 375703
Variables: 15
cat_cols: 0
num_cols: 12
cat_but_car: 3
num_but_cat: 0


In [11]:
# --- [CELL 10]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 11}
df2=df.copy()

In [12]:
# --- [CELL 11]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 12}
sc = MinMaxScaler((0, 1))
df2[num_cols] = sc.fit_transform(df2[num_cols])

In [13]:
# --- [CELL 12]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 13}
kmeans = KMeans(n_clusters=30, n_init="auto").fit(df2[["TotalTime","Calories","SugarContent"]])

In [14]:
# --- [CELL 13]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 14}
clusters_kmeans = kmeans.labels_
clusters_kmeans

array([14, 14,  6, ..., 11,  1, 18], dtype=int32)

In [15]:
# --- [CELL 14]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 15}
df["kmeans_cluster"] = clusters_kmeans
df["kmeans_cluster"]= df["kmeans_cluster"] + 1
df.head()

,Name,CookTime,PrepTime,TotalTime,RecipeIngredientParts,Calories,FatContent,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeInstructions,kmeans_cluster
0,Low-Fat Berry Blue Frozen Dessert,1200,45,1485.00,"c(""blueberries"", ""granulated sugar"", ""vanilla ...",170.90,2.50,1.30,8.00,29.80,37.10,3.60,30.20,3.20,"c(""Toss 2 cups berries with sugar."", ""Let stan...",15
1,Carina's Tofu-Vegetable Kebabs,20,600,1460.00,"c(""extra firm tofu"", ""eggplant"", ""zucchini"", ""...",536.10,24.00,3.80,0.00,1558.60,64.20,17.30,32.10,29.30,"c(""Drain the tofu, carefully squeezing out exc...",15
2,Cabbage Soup,30,20,50.00,"c(""plain tomato juice"", ""cabbage"", ""onion"", ""c...",103.60,0.40,0.10,0.00,959.30,25.10,4.80,17.70,4.30,"c(""Mix everything together and bring to a boil...",7
3,Buttermilk Pie With Gingersnap Crumb Crust,50,30,80.00,"c(""sugar"", ""margarine"", ""egg"", ""flour"", ""salt""...",228.00,7.10,1.70,24.50,281.80,37.50,0.50,24.70,4.20,"c(""Preheat oven to 350°F."", ""Make pie crust, u...",18
4,A Jad - Cucumber Pickle,0,25,25.00,"c(""rice vinegar"", ""haeo"")",4.30,0.00,0.00,0.00,0.70,1.10,0.20,0.20,0.10,"c(""Slice the cucumber in four lengthwise, then...",19


In [16]:
# --- [CELL 15]: ---
# cell_state: edited
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 16}
# === BEFORE (original) ===
# df.groupby('kmeans_cluster').agg({1: ['count','mean', 'median', 'sum'],
#                                     2: ['count','mean', 'median', 'sum'],
#                                     3: ['count','mean', 'median', 'sum'],
#                                     4: ['count','mean','median', 'sum']})

# === AFTER (edited) ===
# Aggregate numeric features by cluster
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [col for col in numeric_cols if col != "kmeans_cluster"]

agg_dict = {col: ['count', 'mean', 'median', 'sum'] for col in numeric_cols}
df.groupby('kmeans_cluster').agg(agg_dict)

CookTime                         PrepTime                       TotalTime                            Calories                          FatContent                        SaturatedFatContent                       CholesterolContent                          SodiumContent                             CarbohydrateContent                         FiberContent                       SugarContent                        ProteinContent                       
                  count   mean  median      sum    count   mean median     sum     count    mean  median        sum    count   mean median        sum      count  mean median       sum               count mean median       sum              count   mean median        sum         count    mean  median         sum               count   mean median       sum        count  mean median      sum        count  mean median       sum          count  mean median       sum
kmeans_cluster                                                                                                                                                                                                                                                                                                                                                                                                                                                                  
1                  9858  35.77   20.00   352586     9858  19.89  15.00  196028      9858   55.71   40.00  549149.00     9858 648.36 636.40 6391542.80       9858 30.34  28.70 299118.70                9858 8.01   8.60  78953.70               9858 100.49  92.80  990588.20          9858  784.56  716.45  7734160.50                9858  58.09  58.30 572655.40         9858  5.58   4.20 55026.50         9858  3.30   3.30  32485.20           9858 33.85  32.40 333731.40
2                 12827  30.44   20.00   390467    12827  17.86  15.00  229096     12827   48.31   40.00  619667.00    12827 240.99 243.00 3091116.30      12827  9.55   9.40 122497.70               12827 3.92   3.30  50323.10              12827  32.96  22.80  422780.70         12827  233.52  159.00  2995357.10               12827  34.55  33.00 443204.30        12827  2.28   1.50 29221.90        12827 21.62  21.60 277373.60          12827  5.35   3.70  68662.50
3                 37807  23.10   18.00   873434    37807  15.70  10.00  593720     37807   38.84   30.00 1468487.00    37807 227.58 223.70 8604264.50      37807 11.78  11.60 445536.80               37807 4.28   3.60 161751.80              37807  55.93  43.60 2114459.20         37807  390.28  329.50 14755196.50               37807  16.81  13.20 635407.40        37807  2.02   1.20 76312.20        37807  1.17   1.20  44151.20          37807 13.11   9.80 495706.10
4                 17800  35.35   25.00   629162    17800  19.06  15.00  339247     17800   54.47   40.00  969591.00    17800 298.83 297.50 5319129.30      17800 13.09  12.70 233084.70               17800 4.31   3.70  76745.40              17800  50.20  40.80  893555.50         17800  578.42  508.45 10295824.40               17800  30.44  29.20 541875.50        17800  4.57   3.60 81410.90        17800  7.24   7.20 128909.20          17800 15.60  13.50 277622.40
5                  9798  36.29   25.00   355606     9798  19.28  15.00  188920      9798   55.62   45.00  544918.00     9798 313.26 318.55 3069332.30       9798 11.85  12.10 116154.20                9798 4.94   4.60  48408.70               9798  41.47  30.90  406352.90          9798  258.09  194.15  2528788.10                9798  47.23  45.60 462755.00         9798  2.53   1.80 24817.00         9798 31.54  31.50 308980.50           9798  5.85   4.40  57283.30
6                  2337 605.29  360.00  1414554     2337 335.01 600.00  782910      2337 1596.29 1490.00 3730529.50     2337 231.78 184.50  541671.70       2337 10.92   6.50  25531.30                2337 3.13   1.70   7322.20               2337  38.83   6.00   90751.50          2337  418.28  22

In [17]:
numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
assert 'kmeans_cluster' in numeric_columns, 'Expected kmeans_cluster to be a numeric grouping column.'
numeric_columns.remove('kmeans_cluster')
assert len(numeric_columns) >= 5, 'Expected at least five numeric feature columns for this aggregation test.'

agg_result = df.groupby('kmeans_cluster').agg({
    numeric_columns[1]: ['count', 'mean', 'median', 'sum'],
    numeric_columns[2]: ['count', 'mean', 'median', 'sum'],
    numeric_columns[3]: ['count', 'mean', 'median', 'sum'],
    numeric_columns[4]: ['count', 'mean', 'median', 'sum'],
})
assert not agg_result.empty, 'Grouped aggregation should produce a non-empty result.'
assert set(['count', 'mean', 'median', 'sum']).issubset(set(agg_result.columns.get_level_values(1)))

try:
    df.groupby('kmeans_cluster').agg({
        1: ['count', 'mean', 'median', 'sum'],
        2: ['count', 'mean', 'median', 'sum'],
        3: ['count', 'mean', 'median', 'sum'],
        4: ['count', 'mean', 'median', 'sum'],
    })
except KeyError:
    pass
else:
    raise AssertionError('Bug regression: integer-labeled aggregation keys unexpectedly succeeded.')